# DS2 — 반지하 침수 취약지 격자 공간분석

**잠길까 / 2026 안양시 공공데이터·AI 활용 대학생 경진대회**
담당: 차동현 · 산출물: `heatmap_manan.png`, `top10_priority_grid.csv`

---

## 이 분석이 답하는 질문

> **안양시 만안구에서 반지하 침수 위험이 가장 높아 우선 조사해야 할 구역은 어디인가?**

### 왜 필요한가

건축물대장에는 **'반지하'라는 항목이 없다.** 이 때문에 정부와 지자체 모두
관내 반지하 주택의 정확한 분포를 파악하지 못하며, 관련 통계는 모두 추정치다.

서울 강동구는 반지하 6,333호를 **건축지도원 12명을 투입해 26일간** 육안조사했다.
(「반지하주택 전수조사 시행계획」, 2023) 안양시에서 같은 방식은 인력상 불가능하다.

본 분석은 **이미 개방된 공공데이터만으로 우선 조사 구역을 좁힌다.**

### 방법 요약

```
침수 노출 지수  = 정규화(침수등급) + 정규화(침수면적비) + 0.3 × 흔적중복
반지하 추정 지수 = 정규화(세대밀도) × 정규화(노후건축물비율)
우선 조사 지수  = 반지하 추정 × 침수 노출
```

**곱셈을 쓰는 이유** — 반지하가 없는 구역은 침수 위험이 높아도 이 서비스의
대상이 아니다. 이 논리가 곱셈으로 자연스럽게 표현되므로 **임의 가중치를
정할 필요가 없다.** 덧셈이었다면 "왜 그 계수냐"에 답해야 한다.

## 활용 데이터

| 데이터 | 출처 | 파일 | 상태 |
|---|---|---|---|
| 도시침수지도 30년 빈도 | **안양시** | `functions/flood30.json` | 확보 |
| 도시침수지도 50년 빈도 | **안양시** | `functions/flood50.json` | 확보 |
| 침수흔적도 | **안양시** | `functions/trace.json` | 확보 |
| 행정동별 인구·세대현황 | **안양시** | `data/anyang_population.csv` | **선택** |

세대현황 CSV가 없어도 분석은 수행된다. 이 경우 침수 노출만으로 순위를 매기며,
결과에 `_est_source = 미적용(노출만)` 으로 표기된다.

## 1. 환경 준비

In [ ]:
# Colab 에서 실행 시
# !pip install shapely --quiet

import json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import Polygon, box
from shapely.strtree import STRtree

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)

# ── 한글 폰트 ────────────────────────────────────────────
# Colab 에서 처음 실행하면 아래 주석을 풀어 나눔고딕을 설치한다.
# 설치 후 [런타임] > [세션 다시 시작] 을 한 번 해야 적용된다.
# !apt-get install -y fonts-nanum > /dev/null 2>&1
# !fc-cache -fv > /dev/null 2>&1
# !rm -rf ~/.cache/matplotlib

import matplotlib.font_manager as fm

def set_korean_font():
    for cand in ['NanumGothic', 'NanumBarunGothic', 'Malgun Gothic', 'AppleGothic']:
        if any(cand in f.name for f in fm.fontManager.ttflist):
            plt.rcParams['font.family'] = cand
            return cand
    # 시스템에 설치된 나눔 폰트 파일을 직접 등록
    import glob
    for path in glob.glob('/usr/share/fonts/**/Nanum*.ttf', recursive=True):
        fm.fontManager.addfont(path)
        name = fm.FontProperties(fname=path).get_name()
        plt.rcParams['font.family'] = name
        return name
    return None

_font = set_korean_font()
plt.rcParams['axes.unicode_minus'] = False

if _font:
    print(f'한글 폰트: {_font}')
else:
    print('[주의] 한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨집니다.')
    print('       위 셀 상단의 apt-get 주석을 풀고 실행한 뒤 세션을 다시 시작하세요.')

print('준비 완료')

## 2. 경로 설정

Colab 이면 저장소를 클론하거나 파일을 업로드한다.

In [ ]:
# ── Colab 에서 저장소를 클론하는 경우 ──
# !git clone https://github.com/aki-iie/Anyang-gongmojun.git
# %cd Anyang-gongmojun

ROOT = Path('.')
FLOOD30 = ROOT / 'functions' / 'flood30.json'
FLOOD50 = ROOT / 'functions' / 'flood50.json'
TRACE   = ROOT / 'functions' / 'trace.json'
POP     = ROOT / 'data' / 'anyang_population.csv'   # 선택

OUTDIR = ROOT / 'public' / 'assets' / 'data'
OUTDIR.mkdir(parents=True, exist_ok=True)

for p in [FLOOD30, FLOOD50, TRACE]:
    print(f'{"O" if p.exists() else "X"}  {p}')
print(f'{"O" if POP.exists() else "-"}  {POP}  (선택)')

## 3. 침수심 등급 정의

`N330~N334` 는 도시침수지도 30년 빈도 등급이며 숫자가 클수록 깊다.
(`functions/flood.js` 의 `LEVEL` 정의와 동일)

**등급에는 cm 값이 없다.** 판정 엔진(`diagnose.py`)은 침수심을 숫자로 받으므로
대표 침수심으로 환산해야 한다. 환산값의 타당성은 6장에서
침수흔적도의 실측 depth 와 대조해 검증한다.

In [ ]:
SEG_LABEL = {
    'N330': '얕은 침수 예상 구간',
    'N331': '중간 침수 예상 구간',
    'N332': '깊은 침수 예상 구간',
    'N333': '매우 깊은 침수 예상 구간',
    'N334': '가장 깊은 침수 예상 구간',
}
SEG_RANK     = {'N330': 1, 'N331': 2, 'N332': 3, 'N333': 4, 'N334': 5}
SEG_DEPTH_CM = {'N330': 20, 'N331': 40, 'N332': 60, 'N333': 100, 'N334': 150}

CELL_M = 100   # 격자 한 변 (m)

pd.DataFrame([
    {'등급': k, '설명': SEG_LABEL[k], '서열': SEG_RANK[k], '대표침수심(cm)': SEG_DEPTH_CM[k]}
    for k in SEG_LABEL
])

## 4. 데이터 로드

### 파일 구조

`flood30.json`
```
{ "N330": [ [minlon, minlat, maxlon, maxlat, [lon,lat,lon,lat,...]], ... ], ... }
              └─────── bbox ───────┘  └──────── 링 좌표 ────────┘
```

`trace.json`
```
[ {"bbox": [...], "depth": 0.262, "area": 2834.8, "year": "2022",
   "sgg": "41171", "cause": "내수침수", "ring": [lon,lat,...]}, ... ]
```

In [ ]:
def _rings_of(poly_arr):
    """폴리곤 배열에서 링 목록을 뽑는다. 구조가 달라도 깨지지 않게 방어."""
    rings = [v for v in poly_arr[4:] if isinstance(v, list) and len(v) >= 6]
    if not rings and len(poly_arr) >= 4 and all(isinstance(x, (int, float)) for x in poly_arr[:4]):
        x0, y0, x1, y1 = poly_arr[:4]
        rings.append([x0, y0, x1, y0, x1, y1, x0, y1, x0, y0])
    return rings


def _to_polygon(flat):
    """평탄화된 [lon,lat,...] 를 Polygon 으로."""
    if len(flat) < 6:
        return None
    pts = [(flat[i], flat[i+1]) for i in range(0, len(flat)-1, 2)]
    if len(pts) < 3:
        return None
    try:
        p = Polygon(pts)
        if not p.is_valid:
            p = p.buffer(0)
        return p if (p.is_valid and not p.is_empty) else None
    except Exception:
        return None


def load_flood(path):
    raw = json.load(open(path, encoding='utf-8'))
    rows = []
    for seg, polys in raw.items():
        for arr in polys:
            for ring in _rings_of(arr):
                g = _to_polygon(ring)
                if g is not None:
                    rows.append({'seg': seg,
                                 'rank': SEG_RANK.get(seg, 0),
                                 'depth_cm': SEG_DEPTH_CM.get(seg, 0),
                                 'geom': g})
    return pd.DataFrame(rows)


def load_trace(path):
    raw = json.load(open(path, encoding='utf-8'))
    rows = []
    for rec in raw:
        g = _to_polygon(rec.get('ring', []))
        if g is None and rec.get('bbox'):
            g = box(*rec['bbox'])
        if g is None:
            continue
        d = float(rec.get('depth') or 0)
        rows.append({'depth_m': d, 'depth_cm': round(d*100, 1),
                     'area': rec.get('area'), 'year': rec.get('year'),
                     'sgg': rec.get('sgg'), 'cause': rec.get('cause'), 'geom': g})
    return pd.DataFrame(rows)


flood = load_flood(FLOOD30)
trace = load_trace(TRACE)

print(f'침수 폴리곤 {len(flood):,}개')
display(flood.groupby('seg').size().rename('폴리곤 수').to_frame()
        .assign(설명=lambda d: d.index.map(SEG_LABEL)))
print(f'\n침수흔적 {len(trace)}건')
display(trace.drop(columns='geom'))

## 5. 침수흔적도 분석 — 사업계획서 3페이지 근거

여기서 나오는 사실이 서비스 설계의 실증 근거가 된다.

In [ ]:
if len(trace):
    SGG = {'41171': '만안구', '41173': '동안구'}
    t = trace.copy()
    t['구'] = t['sgg'].map(SGG).fillna(t['sgg'])

    print('■ 발생 원인')
    print(t['cause'].value_counts().to_string())
    print('\n■ 발생 연도')
    print(t['year'].value_counts().to_string())
    print('\n■ 구별 건수')
    print(t['구'].value_counts().to_string())
    print(f"\n■ 실측 침수심  최소 {t.depth_cm.min():.1f}cm"
          f"  최대 {t.depth_cm.max():.1f}cm"
          f"  평균 {t.depth_cm.mean():.1f}cm")
    print(f"■ 침수 면적    합계 {t['area'].sum():,.0f} m2")

### 여기서 확인되는 것

1. **원인이 전부 `내수침수`다.** 하천 범람이 아니라 배수 불량과 역류 계열이다.
   본 서비스가 **지표 유입과 내부 역류를 나누어 진단**하는 설계의 실증 근거가 된다.

2. **실측 침수심이 10~26cm 수준이다.** 도시 전체로 보면 경미하지만,
   **반지하의 현관 턱과 창문 하단은 확실히 넘는 높이**다.
   같은 강우가 지상 주택에는 불편이지만 반지하에는 재난이 된다.

3. **기록이 8건뿐이다.** 침수흔적도는 피해자가 직접 신고해야 등재되는 구조이므로,
   기록의 부재가 피해의 부재를 뜻하지 않는다.
   **공식 기록 자체가 반지하를 놓치고 있으며, 이것이 가구 단위 진단이 필요한 이유다.**

## 6. 격자 생성 및 집계

In [ ]:
def make_grid(bounds, cell_m=CELL_M):
    minx, miny, maxx, maxy = bounds
    lat_mid = (miny + maxy) / 2
    deg_lat = cell_m / 111_000.0
    deg_lon = cell_m / (111_320.0 * math.cos(math.radians(lat_mid)))

    rows, gid, y = [], 0, miny
    while y < maxy:
        x = minx
        while x < maxx:
            rows.append({'grid_id': f'G{gid:05d}',
                         'cx': round(x + deg_lon/2, 6),
                         'cy': round(y + deg_lat/2, 6),
                         'geom': box(x, y, x + deg_lon, y + deg_lat)})
            gid += 1
            x += deg_lon
        y += deg_lat
    return pd.DataFrame(rows)


b = np.array([g.bounds for g in flood['geom']])
bounds = (b[:,0].min(), b[:,1].min(), b[:,2].max(), b[:,3].max())
grid = make_grid(bounds)
print(f'분석 범위 {[round(v,4) for v in bounds]}')
print(f'{CELL_M}m 격자 {len(grid):,}개')

In [ ]:
def aggregate(grid, flood, trace):
    """격자별 침수 노출 지표 집계."""
    g = grid.copy()
    for c in ['max_rank','max_depth_cm','flood_ratio','trace_hit','trace_depth_cm']:
        g[c] = 0.0

    fgeoms = list(flood['geom']); ftree = STRtree(fgeoms)
    franks = flood['rank'].to_numpy(); fdepth = flood['depth_cm'].to_numpy()
    has_trace = len(trace) > 0
    if has_trace:
        tgeoms = list(trace['geom']); ttree = STRtree(tgeoms)
        tdepth = trace['depth_cm'].to_numpy()

    for i, cell in enumerate(g['geom']):
        area = cell.area
        idx = ftree.query(cell)
        if len(idx):
            inter, best_rank, best_depth = 0.0, 0, 0.0
            for j in idx:
                fg = fgeoms[j]
                if not cell.intersects(fg):
                    continue
                inter += cell.intersection(fg).area
                if franks[j] > best_rank:
                    best_rank, best_depth = int(franks[j]), float(fdepth[j])
            if best_rank:
                g.at[i,'max_rank']     = best_rank
                g.at[i,'max_depth_cm'] = best_depth
                g.at[i,'flood_ratio']  = min(inter/area, 1.0) if area else 0.0

        if has_trace:
            hit, dmax = 0, 0.0
            for j in ttree.query(cell):
                if cell.intersects(tgeoms[j]):
                    hit = 1
                    dmax = max(dmax, float(tdepth[j]))
            g.at[i,'trace_hit']      = hit
            g.at[i,'trace_depth_cm'] = dmax
    return g


gg = aggregate(grid, flood, trace)
print(f"침수 구역과 겹치는 격자  {(gg.max_rank>0).sum():,} / {len(gg):,}")
print(f"침수흔적 중복 격자       {(gg.trace_hit>0).sum():,}")

## 7. 세대밀도 결합 (선택)

안양시 행정동별 인구·세대현황 CSV 가 있으면 반지하 추정 지수에 반영한다.
없으면 침수 노출만으로 순위를 매긴다.

In [ ]:
hh_density = None
old_ratio  = None

if POP.exists():
    pop = pd.read_csv(POP, encoding='utf-8-sig')
    print('컬럼:', list(pop.columns))
    display(pop.head())
    # TODO: 행정동 경계 SHP 와 격자를 공간 조인해 hh_density 를 만든다.
    #       경계 데이터가 없으면 아래 셀은 건너뛴다.
else:
    print('세대현황 CSV 미확보 - 침수 노출만으로 순위를 산출합니다.')
    print('확보 경로: 공공데이터포털 > 경기도 안양시_행정동별 주민등록 인구 및 세대현황')

## 8. 우선 조사 지수 산출

In [ ]:
def _norm(s):
    lo, hi = s.min(), s.max()
    return pd.Series(np.zeros(len(s)), index=s.index) if hi <= lo else (s - lo) / (hi - lo)


def compute_priority(g, hh_density=None, old_ratio=None):
    d = g.copy()
    # 침수 노출
    d['exposure'] = _norm(d['max_rank']) + _norm(d['flood_ratio']) + 0.3 * d['trace_hit']
    # 반지하 추정
    if hh_density is not None and old_ratio is not None:
        d['banjiha_est'] = _norm(hh_density) * _norm(old_ratio)
        src = '세대밀도 x 노후비율'
    elif hh_density is not None:
        d['banjiha_est'] = _norm(hh_density)
        src = '세대밀도'
    else:
        d['banjiha_est'] = 1.0
        src = '미적용(노출만)'
    d['priority'] = d['banjiha_est'] * d['exposure']
    d.attrs['est_source'] = src
    return d


res = compute_priority(gg, hh_density, old_ratio)
print('반지하 추정 지수 산출 방식:', res.attrs['est_source'])
display(res[['max_rank','flood_ratio','trace_hit','exposure','priority']].describe().round(3))

## 9. 등급 환산값 검증

침수흔적도가 중복되는 격자에서 **등급 대표 침수심**과 **실측 침수심**을 비교한다.
사업계획서에 환산값의 근거로 제시할 표다.

In [ ]:
m = res[res['trace_hit'] == 1]
if len(m):
    rows = []
    for r, sub in m.groupby('max_rank'):
        if r == 0:
            continue
        seg = next((k for k, v in SEG_RANK.items() if v == r), '?')
        rows.append({'등급': seg, '서열': int(r), '격자수': len(sub),
                     '등급대표침수심(cm)': float(sub['max_depth_cm'].iloc[0]),
                     '실측침수심평균(cm)': round(float(sub['trace_depth_cm'].mean()), 1)})
    vdf = pd.DataFrame(rows).sort_values('서열')
    vdf['차이(cm)'] = (vdf['등급대표침수심(cm)'] - vdf['실측침수심평균(cm)']).round(1)
    display(vdf)
    print('\n해석: 등급 대표 침수심이 실측보다 크면 보수적으로 설정된 것이다.')
    print('      본 서비스는 위험을 과소평가하지 않기 위해 보수적 환산을 택했다.')
else:
    print('침수흔적과 겹치는 격자가 없어 검증을 수행하지 못했습니다.')

## 10. 취약지 히트맵

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8), dpi=150)

sc = ax.scatter(res['cx'], res['cy'], c=res['priority'],
                s=9, marker='s', cmap='OrRd', linewidths=0)

# 침수흔적 위치 표시
if len(trace):
    tc = np.array([[g.centroid.x, g.centroid.y] for g in trace['geom']])
    ax.scatter(tc[:,0], tc[:,1], s=70, facecolors='none',
               edgecolors='#1f3864', linewidths=1.6, label='침수흔적 실측 지점')

# 상위 10곳
top10 = res.nlargest(10, 'priority')
ax.scatter(top10['cx'], top10['cy'], s=90, facecolors='none',
           edgecolors='#0b6b3a', linewidths=1.8, label='우선 조사 상위 10')

cb = fig.colorbar(sc, ax=ax, shrink=0.85)
cb.set_label('우선 조사 지수', fontsize=10)
ax.set_title(f'안양시 반지하 침수 취약지 추정  ({CELL_M}m 격자, 30년 빈도 기준)',
             fontsize=13, pad=12)
ax.set_xlabel('경도'); ax.set_ylabel('위도')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.grid(alpha=0.15, linewidth=0.4)
fig.tight_layout()

out_png = OUTDIR / 'heatmap_manan.png'
fig.savefig(out_png, dpi=150, bbox_inches='tight')
print('저장:', out_png)
plt.show()

## 11. 우선 조사 구역 Top 10

In [ ]:
top = res.nlargest(10, 'priority').copy()
top['순위'] = range(1, len(top)+1)
top['등급'] = top['max_rank'].map({v: k for k, v in SEG_RANK.items()}).fillna('-')
top['등급설명'] = top['등급'].map(SEG_LABEL).fillna('-')

cols = ['순위','grid_id','cx','cy','등급','등급설명',
        'max_depth_cm','flood_ratio','trace_hit','exposure','priority']
out = top[cols].rename(columns={
    'grid_id':'격자ID','cx':'중심경도','cy':'중심위도',
    'max_depth_cm':'대표침수심(cm)','flood_ratio':'침수면적비',
    'trace_hit':'흔적중복','exposure':'침수노출지수','priority':'우선조사지수'})
out[['침수면적비','침수노출지수','우선조사지수']] = \
    out[['침수면적비','침수노출지수','우선조사지수']].round(4)

out_csv = OUTDIR / 'top10_priority_grid.csv'
out.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('저장:', out_csv)
display(out)

## 12. 전체 격자 결과 저장 (대시보드 연동용)

In [ ]:
full = res[res['max_rank'] > 0][
    ['grid_id','cx','cy','max_rank','max_depth_cm',
     'flood_ratio','trace_hit','trace_depth_cm','exposure','priority']
].copy()
full[['flood_ratio','exposure','priority']] = full[['flood_ratio','exposure','priority']].round(4)
full = full.sort_values('priority', ascending=False)

out_all = OUTDIR / 'grid_priority_all.csv'
full.to_csv(out_all, index=False, encoding='utf-8-sig')
print(f'저장: {out_all}  ({len(full):,}행)')
display(full.head(20))

## 13. 사업계획서용 요약

In [ ]:
n_grid   = len(res)
n_flood  = int((res.max_rank > 0).sum())
n_trace  = int((res.trace_hit > 0).sum())
top1     = res.nlargest(1, 'priority').iloc[0]

print(f"""
── DS2 분석 요약 ─────────────────────────────────

  분석 단위       {CELL_M}m 격자 {n_grid:,}개
  침수 구역 포함   {n_flood:,}개 격자 ({n_flood/n_grid*100:.1f}%)
  흔적 중복        {n_trace}개 격자
  원본 폴리곤      {len(flood):,}개 (도시침수지도 30년 빈도)
  침수흔적 기록    {len(trace)}건 (2022년, 전부 내수침수)

  최우선 구역      {top1.grid_id}
                  경위도 ({top1.cx:.5f}, {top1.cy:.5f})
                  우선조사지수 {top1.priority:.4f}

  반지하 추정      {res.attrs['est_source']}

  산출물
    heatmap_manan.png
    top10_priority_grid.csv
    grid_priority_all.csv
──────────────────────────────────────────────────
""")

---

## 한계와 대응

| 한계 | 대응 |
|---|---|
| 건축물대장의 반지하 항목 부재 | 세대밀도와 노후건축물 비율로 **추정**. 추정임을 명시 |
| 침수흔적 기록 8건 | 통계적 검증이 아니라 **등급 환산값 대조**에만 사용 |
| 등급의 cm 환산 | 보수적으로 설정. 9장에서 실측과 대조해 근거 제시 |
| 세대현황 CSV 미확보 | 침수 노출만으로도 순위 산출 가능. CSV 확보 시 재실행 |

## 다음 단계

1. 안양시 행정동별 인구·세대현황 CSV 확보 후 7장 재실행
2. 행정동 경계 SHP 로 격자-행정동 공간 조인
3. 산출물을 `docs/` 로 옮겨 사업계획서 7페이지에 삽입